# CO3133 A2 - Food-101 data preparation (Kaggle, **CPU** notebook)

Runs without GPU quota. Produces the pre-resized copy `food-101-256/` (shorter side 256 px, ≈ 2 GB)
that the GPU notebook reads.

1. *Settings → Accelerator: None*, *Internet: On*.
2. Optional: *Add input* → a Kaggle copy of Food-101 that keeps the official layout
   (`meta/classes.txt`, `images/<class>/<id>.jpg`). Otherwise the official archive is downloaded (~5 GB).
3. *Save Version → Save & Run All (Commit)*. When it finishes, the output folder `food-101-256` can be
   added to the GPU notebook with *Add input → Your Work → this notebook*.

The split lists, normalization statistics and image index are **already committed** in the repository
(`data/splits/`, `data/meta/`), so this notebook only resizes and verifies.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/pbt245/CO3113-A2.git"
REF = "main"            # set a commit hash or tag for the final runs (traceability)
REPO = "/kaggle/working/CO3113-A2"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", REF], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import glob
def find_food101(pattern_root="/kaggle/input"):
    """Return every folder under /kaggle/input that has meta/classes.txt and images/."""
    hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{pattern_root}/**/meta/classes.txt", recursive=True)]
    return [h for h in hits if os.path.isdir(os.path.join(h, "images"))]
print(find_food101())

In [ ]:
found = find_food101()
if found:
    RAW = found[0]
else:
    subprocess.run([sys.executable, "scripts/download_data.py", "--raw-dir", "/kaggle/temp/raw"], check=True)
    RAW = "/kaggle/temp/raw/food-101"
print("raw Food-101:", RAW)

In [ ]:
DST = "/kaggle/working/food-101-256"
!python scripts/preresize.py --src {RAW} --dst {DST} --workers 4

In [ ]:
# verify: every entry of the committed split exists in the resized copy
from pathlib import Path
missing = [e for s in ("train", "val", "test") for e in Path(f"data/splits/{s}.txt").read_text().split()
           if not Path(f"{DST}/images/{e}.jpg").exists()]
print("missing images:", len(missing))
assert not missing

In [ ]:
# keep only the resized copy in the notebook output
import shutil
os.chdir("/kaggle/working")
shutil.rmtree(REPO)
!du -sh {DST}